# Repeat Ground-Track Orbit Design
**From two-body to J2+J3 with Broyden refinement**

This notebook builds the SMA solution progressively:

| Step | Model | Unknowns solved |
|------|-------|-----------------|
| 1 | Two-body | $a_{J_1}$ (seed) |
| 2 | +J2 secular | $a_{J_2}$ (iterative fixed-point) |
| 3 | +J3 frozen | $e_f$ (analytical) |
| 4 | J2+J3 coupled | $(a^*, e^*)$ via 2-D Broyden |

**Reference:** D'Amico & Montenbruck, *TerraSAR-X Reference Orbit Design*, 2004.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml, pathlib, sys

sys.path.insert(0, str(pathlib.Path().resolve().parent))

# Load config
cfg  = yaml.safe_load((pathlib.Path().resolve().parent / 'config.yaml').read_text())
nfg  = cfg['neqfro']

# ── Constants (EGM96 / WGS84) ────────────────────────────────────────────────
MU          =  3.986004418e14      # m³ s⁻²
RE          =  6_378_137.0         # m
J2          =  1.08262668e-3
J3          = -2.5415e-6
OMEGA_E     =  7.2921150e-5        # rad/s  (sidereal rotation)
OMEGA_SUN   =  2*np.pi / (365.25 * 86400.0)  # rad/s  (solar precession)
T_SIDEREAL  =  86_164.0905         # s

# ── Mission parameters from config ───────────────────────────────────────────
k = int(nfg['repeat_k'])           # satellite revolutions per cycle
q = float(nfg['repeat_q_days'])    # sidereal days per cycle
orbit_type = nfg['orbit_type']
i_deg_cfg  = nfg.get('inclination_deg', 10.0)

print(f'Repeat cycle : {k} orbits / {q} sidereal days')
print(f'Orbit type   : {orbit_type}')
print(f'Config incl. : {i_deg_cfg} deg')

---
## Step 1 — Two-Body SMA ($a_{J_1}$)

The two-body potential gives the Keplerian repeat condition:
$$k \cdot T_{\text{orbit}} = q \cdot T_{\text{sidereal}} \implies n_0 = \frac{k \,\omega_E}{q}$$

Inverting Kepler's third law:
$$a_{J_1} = \left(\frac{P}{2\pi}\sqrt{GM_\oplus}\right)^{2/3} = \left(\frac{\mu}{n_0^2}\right)^{1/3}$$

This is the **zero-perturbation seed** — the SMA if Earth were a perfect sphere.

In [ ]:
n0   = k * OMEGA_E / q
a_J1 = (MU / n0**2) ** (1/3)
P    = 2*np.pi / n0

print(f'n₀      = {n0:.6e} rad/s')
print(f'Period  = {P/60:.3f} min')
print(f'a_J1    = {a_J1/1e3:.3f} km   (alt = {(a_J1-RE)/1e3:.1f} km)')

---
## Step 2 — J2-Corrected SMA ($a_{J_2}$)

J2 introduces three secular rates that shift the effective angular velocity of the sub-satellite point:

| Rate | Expression | Effect on repeat |
|------|-----------|------------------|
| $\dot{\Omega}$ | $-\frac{3nJ_2R_E^2}{2p^2}\cos i$ | Changes Earth's effective rotation |
| $\dot{\omega}$ | $\frac{3nJ_2R_E^2}{4p^2}(5\cos^2 i-1)$ | Perigee precesses, shifts argument of latitude |
| $\delta\dot{M}$ | $\frac{3nJ_2R_E^2}{4p^2}\sqrt{1-e^2}(3\cos^2 i-1)$ | Mean anomaly rate corrected by J2 |

The **effective mean motion** combines $\dot{\omega}$ and $\delta\dot{M}$:
$$n_{\text{eff}} = n\,(1+\gamma), \quad \gamma = \frac{3J_2R_E^2}{4p^2}\left[\sqrt{1-e^2}(3\cos^2 i-1)+(5\cos^2 i-1)\right]$$

The **governing repeat condition** becomes:
$$n_{\text{eff}} = \frac{k}{q}(\omega_E - \dot{\Omega}) \implies n = \frac{(k/q)(\omega_E-\dot{\Omega})}{1+\gamma}$$

Since $\dot{\Omega}$ and $\gamma$ themselves depend on $a$, this is solved by fixed-point iteration.

In [ ]:
def nodal_regression(a, e, i):
    n = np.sqrt(MU / a**3)
    return -(3/2) * n * J2 * (RE/a)**2 * np.cos(i) / (1-e**2)**2

def n_eff_factor(a, e, i):
    """γ such that n_eff = n*(1+γ)  (combines δṀ_J2 and ω̇)"""
    eta2 = 1 - e**2
    p2   = (a * eta2)**2
    c2i  = np.cos(i)**2
    return (3*J2*RE**2 / (4*p2)) * (np.sqrt(eta2)*(3*c2i-1) + (5*c2i-1))

def sun_sync_inclination(a, e=0.0):
    cos_i = (-2*OMEGA_SUN * a**3.5 * (1-e**2)**2
             / (3*np.sqrt(MU) * J2 * RE**2))
    return np.arccos(np.clip(cos_i, -1, 1))

def solve_sma_J2(k, q, i_rad, e=1e-3, tol=1.0, max_iter=100):
    """Fixed-point iteration for the J2 repeat SMA."""
    a = (MU / (k*OMEGA_E/q)**2) ** (1/3)   # two-body seed
    history = [a]
    for _ in range(max_iter):
        dOmega = nodal_regression(a, e, i_rad)
        gamma  = n_eff_factor(a, e, i_rad)
        n_req  = (k * (OMEGA_E - dOmega) / q) / (1 + gamma)
        a_new  = (MU / n_req**2) ** (1/3)
        history.append(a_new)
        if abs(a_new - a) < tol:
            return a_new, history
        a = a_new
    return a, history

# ── Solve for inclination depending on orbit type ─────────────────────────────
if orbit_type == 'sun_synchronous':
    i_rad = np.radians(97.4)  # seed
    for _ in range(50):
        a_J2, _ = solve_sma_J2(k, q, i_rad)
        i_new = sun_sync_inclination(a_J2)
        if abs(i_new - i_rad) < 1e-9: break
        i_rad = i_new
else:
    i_rad = np.radians(i_deg_cfg)
    a_J2, _ = solve_sma_J2(k, q, i_rad)

print(f'Inclination : {np.degrees(i_rad):.4f} deg')
print(f'a_J1        = {a_J1/1e3:.3f} km')
print(f'a_J2        = {a_J2/1e3:.3f} km   (delta = {(a_J2-a_J1):.1f} m from J2)')

In [ ]:
# ── Visualise convergence of the fixed-point iteration ────────────────────────
_, hist = solve_sma_J2(k, q, i_rad)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(hist, 'o-', ms=5)
ax.axhline(a_J2, color='r', ls='--', label=f'Converged {a_J2/1e3:.3f} km')
ax.set_xlabel('Iteration')
ax.set_ylabel('SMA [m]')
ax.set_title('Fixed-point convergence of J2 repeat condition')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 3 — J3 Frozen Eccentricity

J3 (odd zonal) does **not** produce secular RAAN or argument-of-perigee drift.  
Instead, it drives a secular evolution of eccentricity unless the orbit is **frozen**.

The secular balance between J2 ($\dot{\omega} \neq 0$) and J3 ($\dot{e} \neq 0$) yields a stationary eccentricity vector at $\omega = 90°$ (Coffey-Deprit formula):

$$e_f = -\frac{J_3}{2J_2}\frac{R_E}{a}\sin i$$

This $e_f$ depends on $a$, so Steps 2 and 3 are **coupled**: the frozen $e_f$ feeds back into the J2 repeat condition through $\gamma(a, e_f)$ and $\dot{\Omega}(a, e_f)$.

In [ ]:
def frozen_eccentricity(a, i_rad):
    """J3 frozen eccentricity at ω = 90° (Coffey-Deprit)."""
    return -(J3 / (2*J2)) * (RE / a) * np.sin(i_rad)

e_f = frozen_eccentricity(a_J2, i_rad)

print(f'Frozen eccentricity e_f = {e_f:.6e}')
print(f'  (at ω = 90°, J3 perturbation is stationary)')
print()
print('Feedback: re-solve a_J2 using the frozen e_f...')
a_J2J3, _ = solve_sma_J2(k, q, i_rad, e=e_f)
e_f2 = frozen_eccentricity(a_J2J3, i_rad)
print(f'  a_J2 (e=1e-3) = {a_J2/1e3:.4f} km')
print(f'  a_J2 (e=e_f)  = {a_J2J3/1e3:.4f} km   delta = {(a_J2J3-a_J2):.2f} m')
print(f'  e_f (updated) = {e_f2:.6e}   delta = {(e_f2-e_f):.2e}')

---
## Step 4 — Coupled (a, e) via 2-D Broyden

The feedback above shows that $a$ and $e_f$ shift when the other changes.  
We can formulate this as a **2-D root-finding** problem:

$$\mathbf{F}(a,\, e) = \begin{pmatrix} F_1 \\ F_2 \end{pmatrix} = \begin{pmatrix} a - a_{\text{repeat}}(e) \\ e - e_f(a) \end{pmatrix} = \mathbf{0}$$

where $a_{\text{repeat}}(e)$ is the repeat SMA at eccentricity $e$, and $e_f(a)$ is the frozen eccentricity at SMA $a$.

**Broyden's method** avoids recomputing the full Jacobian at every step — it performs a rank-1 update from successive residuals, keeping convergence super-linear at the cost of only two function evaluations per iteration.

In [ ]:
def residual(x):
    """F(a, e) = [a - a_repeat(e), e - e_frozen(a)]."""
    a, e = x
    e = max(e, 1e-8)                          # prevent negative eccentricity
    a_rep, _ = solve_sma_J2(k, q, i_rad, e=e) # repeat condition
    e_frz    = frozen_eccentricity(a, i_rad)   # J3 frozen condition
    return np.array([a - a_rep, e - e_frz])

def broyden_2d(F, x0, tol=1e-3, max_iter=30):
    """2-D Broyden's Good Method (rank-1 inverse Jacobian update)."""
    x  = np.array(x0, dtype=float)
    f  = F(x)
    dx = np.array([10.0, 1e-6])               # finite-difference step

    # Seed the inverse Jacobian numerically
    J_inv = np.zeros((2, 2))
    for j in range(2):
        xp     = x.copy(); xp[j] += dx[j]
        J_inv[:, j] = (F(xp) - f) / dx[j]
    J_inv = np.linalg.inv(J_inv)

    history = [x.copy()]
    for i in range(max_iter):
        step  = -J_inv @ f
        x_new = x + step
        x_new[1] = max(x_new[1], 1e-8)       # eccentricity >= 0
        f_new = F(x_new)

        # Broyden rank-1 update
        df = f_new - f
        u  = step - J_inv @ df
        v  = step @ df
        if abs(v) > 1e-30:
            J_inv += np.outer(u, step @ J_inv) / v

        x, f = x_new, f_new
        history.append(x.copy())
        if np.linalg.norm(f) < tol:
            break

    return x, np.array(history)

# ── Solve ─────────────────────────────────────────────────────────────────────
x0   = np.array([a_J2, e_f])
(a_star, e_star), hist2d = broyden_2d(residual, x0)

print(f'Broyden converged in {len(hist2d)-1} iterations')
print(f'  a* = {a_star/1e3:.4f} km')
print(f'  e* = {e_star:.6e}')
print()
print('Comparison:')
print(f'  a_J2 (e=1e-3 seed)  = {a_J2/1e3:.4f} km')
print(f'  a_J2 (e=e_f)        = {a_J2J3/1e3:.4f} km')
print(f'  a* (Broyden coupled)= {a_star/1e3:.4f} km')

In [ ]:
# ── Visualise Broyden path in (a, e) space ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

iters = np.arange(len(hist2d))
axes[0].plot(iters, (hist2d[:, 0] - a_star), 'o-', ms=5)
axes[0].axhline(0, color='r', ls='--')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('a − a* [m]')
axes[0].set_title('SMA convergence')

axes[1].plot(iters, (hist2d[:, 1] - e_star), 's-', ms=5, color='tab:orange')
axes[1].axhline(0, color='r', ls='--')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('e − e* ')
axes[1].set_title('Eccentricity convergence')

plt.suptitle('2-D Broyden: coupled (a, e) convergence', y=1.02)
plt.tight_layout()
plt.show()

---
## Is HPOP Needed Inside the Broyden Step?

**Short answer: No, for a J2+J3 model. Yes, for a high-fidelity design.**

The table below shows what each residual function evaluates:

| Broyden residual source | Forces included | Typical Δa residual | Runtime / call |
|------------------------|----------------|---------------------|----------------|
| Analytical J2+J3 (this notebook) | J2 + J3 only | < 1 m (self-consistent) | < 1 ms |
| J3 NumericalPropagator (`j3_propagator.py`) | J2 + J3 (numerical) | O(1–10 m) | ~1 s |
| HPOP (`hpop.py`, degree 70) | J2–J70, drag, 3rd-body, SRP | O(10–100 m) vs J2+J3 seed | ~10–30 s |

### Why HPOP matters
The analytical J2+J3 solution is **exact within its own model** — the Broyden residual $\mathbf{F}(a^*, e^*)$ is already zero analytically. Running Broyden on the analytical residual converges immediately because it is the exact fixed point.

HPOP is required when:
1. **Higher-order zonals (J4–J70)** shift the mean SMA by tens of metres.
2. **Atmospheric drag** causes slow secular decay not captured analytically.
3. **Third-body effects** (Sun/Moon) produce long-period oscillations in $e$ not represented by the Coffey-Deprit formula.
4. **The final design must close numerically**, not just analytically — the ground-track must actually repeat under the full force model.

### Recommended workflow
```
1. Analytical J2+J3  →  (a₀, e₀)  [this notebook, < 1 s]
       ↓  use as initial guess
2. Broyden + HPOP    →  (a*, e*)  [design_neqfro.py, ~5–15 min]
       ↓  closes the repeat condition under the full force model
3. Rosengren         →  e_frozen  [find minimum phase-space radius]
```

The analytical solution typically lies within **50–200 m** of the HPOP-converged SMA for LEO orbits, so Broyden needs only 3–6 HPOP evaluations to converge — this is why a good analytical seed matters.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 60)
print('  Design Summary (analytical J2+J3)')
print('=' * 60)
print(f'  Repeat cycle       : {k} orbits / {q} sidereal days')
print(f'  Inclination        : {np.degrees(i_rad):.4f} deg')
print(f'  SMA (two-body)     : {a_J1/1e3:.3f} km')
print(f'  SMA (J2, e=1e-3)   : {a_J2/1e3:.3f} km  (+{(a_J2-a_J1):.0f} m)')
print(f'  SMA (J2+J3 coupled): {a_star/1e3:.3f} km  (+{(a_star-a_J1):.0f} m from two-body)')
print(f'  Frozen eccentricity: {e_star:.6e}')
print(f'  Arg. of perigee    : 90.0 deg  (frozen condition)')
print(f'  Altitude           : {(a_star - RE)/1e3:.1f} km')
print('=' * 60)
print()
print('Next step: run design_neqfro.py with run_broyden: true')
print('to refine (a, e) against the full HPOP force model.')